# Message-driven FIPS 205 signing-order PRF capture

For every experimental signature, the host generates a reproducible message,
constructs the FIPS 205 pure-signature input, evaluates SHA-2 `PRF_msg` and
`H_msg`, extracts `md`, `idx_tree`, `idx_leaf`, and FORS indices, and enumerates
the WOTS/FORS address-PRF calls in signing-pseudocode order.

The target executes **one isolated PRF per trace**. This is **not** an actual
whole-signature power trace. The host derives the corresponding `PK.root` from
the selected `SK.seed` and `PK.seed` and caches it.

In `hedged` mode, additional randomness is deterministically derived from a
public experiment seed for replay; it is not production cryptographic entropy.

The append-safe workflow checks existing data and captures only missing
signature IDs. Partial/incompatible data causes an error rather than an
unconditional overwrite.


## 1. Imports and paths

Specify the firmware directory outside the public repository.


In [ ]:
from __future__ import annotations

import importlib
import os
import sys
from pathlib import Path

import chipwhisperer as cw

# Run this notebook from the repository root or the notebooks/ folder.
WORKDIR = Path.cwd().resolve()
PROJECT_ROOT = WORKDIR if (WORKDIR / "src").is_dir() else WORKDIR.parent
MODULE_DIR = PROJECT_ROOT / "src"
if not MODULE_DIR.is_dir():
    raise FileNotFoundError("Cannot find src/. Start Jupyter from the repository.")
if str(MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(MODULE_DIR))

# Override these paths locally via environment variables (do not commit paths).
FIRMWARE_DIR = Path(os.environ.get(
    "SLH_DSA_FIRMWARE_DIR",
    str(PROJECT_ROOT / "firmware" / "simpleserial-SLH-DSA"),
)).expanduser()
RESULTS_BASE_DIR = Path(os.environ.get(
    "SLH_DSA_RESULTS_DIR", str(PROJECT_ROOT / "results"),
)).expanduser()

import prf_capture_core as live
live = importlib.reload(live)

import random_address_capture as autobatch
autobatch = importlib.reload(autobatch)

print("Capture helper:", live.CAPTURE_HELPER_VERSION)
print("Batch helper:", autobatch.BATCH_HELPER_VERSION)
print("Firmware directory:", FIRMWARE_DIR)
print("Results directory:", RESULTS_BASE_DIR)

import signature_prf_sequence as seqcap
seqcap = importlib.reload(seqcap)

import signature_message_path as msgcap
msgcap = importlib.reload(msgcap)

import signature_sequence_capture as sigbatch
sigbatch = importlib.reload(sigbatch)

assert hasattr(seqcap, "encode_signature_metadata_rows_uint64")
assert "uint64" in msgcap.MESSAGE_CAPTURE_HELPER_VERSION
print("Sequence helper:", seqcap.SIGNATURE_SEQUENCE_HELPER_VERSION)
print("Message helper:", msgcap.MESSAGE_CAPTURE_HELPER_VERSION)
print("Signature metadata dtype:", seqcap.SIGNATURE_META_DTYPE)


## 2. Metadata format smoke test

The metadata uses unsigned 64-bit integers to preserve large `idx_tree` values for 256f. Optional missing fields use a per-column sentinel.


In [ ]:
# uint64 metadata smoke test (no ChipWhisperer acquisition)
# Simulate a SLH-DSA-256f tree index above signed int64 maximum.
import numpy as np

max_tree = (1 << 64) - 1
_dummy_case = live.Case(
    kind=live.KIND_WOTS, layer=0, tree=max_tree,
    keypair=0, chain=0, hash=0, tree_height=0, tree_index=0,
    addr=bytes(32),
)
_dummy_call = seqcap.SignaturePRFCall(
    case=_dummy_case, signature_id=2, algorithm_call_index=0,
    phase_code=seqcap.PHASE_XMSS_TREEHASH,
    tree_number=-1, selected_index=13,
)
_check = seqcap.encode_signature_metadata_rows_uint64([_dummy_call])
assert _check.dtype == np.dtype("uint64")
assert int(_check[0, 5]) == max_tree
assert int(_check[0, 10]) == max_tree  # -1 optional sentinel
assert int(_check[0, 11]) == 13
print("uint64 metadata smoke test: PASS")


## 3. Configure signing-path capture

The defaults capture one message-derived signing path per secret seed. Update the per-parameter targets and local paths for your environment.


In [ ]:
SCOPETYPE = "OPENADC"
PLATFORM = "CW308_STM32F3"
CRYPTO_TARGET = "MBEDTLS"
HASH_MODE = "sha2"

# A safe starting point; add all six parameter sets as needed.
PARAM_SETS = ["128f"]
COMPILER_OPTS = ["Os"]
BUILD_FIRMWARE = True
PROGRAM_TARGET = True

# The target is a count PER secret seed, not a total across seeds.
SIGNATURES_PER_PARAM_SET = {p: 1 for p in PARAM_SETS}
NUM_SECRET_SEEDS = 3

MESSAGE_LENGTH = 32
MESSAGE_SEED = 777
SIGNING_VARIANT = "hedged"  # "hedged" or "deterministic"
ADDRND_SEED = 778
CONTEXT = b""
CAPTURE_PHASES = None  # All four phases, or a selected subset.

# The demo seed differs from the private experimental master seed.
# For an existing dataset, the local setting must match its original manifest.
SECRET_SEED_MASTER = int(os.environ.get("SLH_DSA_SECRET_SEED_MASTER", "424242"))
REGENERATE_SHARED_SEED_MANIFEST = False
FORCE_RECOMPUTE_PK_ROOT = False
RESUME = True
CONTINUE_ON_ERROR = False

CALIBRATION_KIND = "wots"
numberOfPoints = -1
MAX_SAMPLES = 131124
AUTO_CALIBRATE_POINTS = True
CALIBRATION_MARGIN = 0
presamples = 0
gain = 15
capture_timeout_s = 10
ss_timeout = 250
VERIFY_ADDR_EACH_TIME = False
VERIFY_RESULT = True
VERIFY_FIRST_N = 5
FLUSH_EVERY = 1000


## 4. Preview data reuse and additional captures

This is a file-count preview, not a verification of file contents. The actual capture performs strict compatibility checks first.


In [ ]:
plan = sigbatch.preview_incremental_multiseed_capture_plan(
    live=live,
    results_base_dir=RESULTS_BASE_DIR,
    param_sets=PARAM_SETS,
    compiler_opts=COMPILER_OPTS,
    signatures_per_param_set=SIGNATURES_PER_PARAM_SET,
    num_secret_seeds=NUM_SECRET_SEEDS,
    capture_phases=CAPTURE_PHASES,
    hash_mode=HASH_MODE,
)

print("Parameter / compiler / seed | target | existing candidate | add (if valid) | PRF calls to add")
print("-" * 108)
for row in plan:
    print(
        f"{row['parameter_set']:4s}/{row['compiler_opt']:2s}/seed_{row['seed_id']:03d} | "
        f"{row['requested_signatures']:3d} | "
        f"{row['existing_candidates']:3d} | "
        f"{row['additional_if_valid']:3d} | "
        f"{row['additional_prf_calls_if_valid']:12,d}"
    )
    if row['partial_or_incompatible_candidate_ids']:
        print(" WARNING partial candidate IDs:", row['partial_or_incompatible_candidate_ids'])
print()
print("Total additional signatures (assuming candidate files valid):", sum(r['additional_if_valid'] for r in plan))
print("Total additional PRF calls (assuming candidate files valid):", f"{sum(r['additional_prf_calls_if_valid'] for r in plan):,}")


## 5. Connect the ChipWhisperer scope and target


In [ ]:
scope = cw.scope()
scope.default_setup()
scope.gain.db = gain
scope.adc.samples = int(MAX_SAMPLES)
scope.adc.presamples = int(presamples)
scope.adc.timeout = int(capture_timeout_s)

target = cw.target(scope, cw.targets.SimpleSerial)

print("scope/target connected")
print("gain          :", scope.gain.db)
print("initial sample:", scope.adc.samples)


## 6. Append-safe multiseed capture

A completed signature is reused. Incompatible or partially saved files are not silently overwritten.


In [ ]:
batch_state = sigbatch.run_all_parameter_random_message_signature_captures_multiseed(
    scope,
    target,
    live,

    firmware_dir=FIRMWARE_DIR,
    results_base_dir=RESULTS_BASE_DIR,

    platform=PLATFORM,
    crypto_target=CRYPTO_TARGET,
    hash_mode=HASH_MODE,

    param_sets=PARAM_SETS,
    compiler_opts=COMPILER_OPTS,
    signatures_per_param_set=SIGNATURES_PER_PARAM_SET,
    capture_phases=CAPTURE_PHASES,

    message_seed=MESSAGE_SEED,
    message_length=MESSAGE_LENGTH,
    signing_variant=SIGNING_VARIANT,
    addrnd_seed=ADDRND_SEED,
    context=CONTEXT,

    num_secret_seeds=NUM_SECRET_SEEDS,
    secret_seed_master=SECRET_SEED_MASTER,
    regenerate_shared_seed_manifest=REGENERATE_SHARED_SEED_MANIFEST,
    force_recompute_pk_root=FORCE_RECOMPUTE_PK_ROOT,

    build_firmware=BUILD_FIRMWARE,
    program_target=PROGRAM_TARGET,

    calibration_kind=CALIBRATION_KIND,
    number_of_points=numberOfPoints,
    max_samples=MAX_SAMPLES,
    auto_calibrate_points=AUTO_CALIBRATE_POINTS,
    calibration_margin=CALIBRATION_MARGIN,

    timeout=ss_timeout,
    verify_addr_each_time=VERIFY_ADDR_EACH_TIME,
    verify_result=VERIFY_RESULT,
    verify_first_n=VERIFY_FIRST_N,
    flush_every=FLUSH_EVERY,

    resume=RESUME,
    continue_on_error=CONTINUE_ON_ERROR,
)

print("batch finished")


## 7. Saved results

Example: `results/SLH_DSA_128f_sha2_message_signature_sequence/Os/seed_000/signature_000/`. Each signature directory contains trace arrays, addresses, metadata, and a JSON description of the message/path. Seed manifests and root caches are local experimental data and are excluded from Git.


## 8. Verify completed signature counts


In [ ]:
from pathlib import Path
import json

print("=== Signature counts per secret seed ===")
for p in PARAM_SETS:
    for opt in COMPILER_OPTS:
        for seed_id in range(NUM_SECRET_SEEDS):
            seed_dir = (
                RESULTS_BASE_DIR
                / f"SLH_DSA_{p}_{HASH_MODE}_message_signature_sequence"
                / opt
                / f"seed_{seed_id:03d}"
            )
            mf = seed_dir / "signature_sequence_capture_manifest.json"
            if not mf.is_file():
                print(f"{p}/{opt}/seed_{seed_id:03d}: no manifest")
                continue
            data = json.loads(mf.read_text(encoding="utf-8"))
            rows = [r for r in data.get("completed", [])
                    if str(r.get("status", "")).startswith("completed")]
            ids = sorted(int(r['signature_id']) for r in rows)
            print(f"{p}/{opt}/seed_{seed_id:03d}: {len(ids)} signatures, IDs={ids}")
